In [ ]:
print("[cell] 01 run parameters (kapi --env): environment variables of this r", flush=True)
# run parameters (kapi --env): environment variables of this run
import os
os.environ['RFS_UPLOAD'] = 'phanththui/flag2027-feats-rfs-b-p'
os.environ['RFS_BUDGET_H'] = '8.0'

# FLAG 2027 — NB-162b: RFS-01 step 1, the RF-L2-092 feature blocks for our people store (lane auxb)

**Why.** The user decided (3/11, nhat ky 360): the final system = F1s + **RFS**, where RFS is the RF-092 recipe (CB 18.61) **re-learned on our filtered people
store**. We take only RF's code / technique, never its learned artifacts. This notebook only **extracts** the blocks RF's projector and mixer need, with
**RF's own extractors** (`flag_final_core_092.py`, embedded below, sha256 asserted). No Bengali filtering at extraction (step 2 filters with `lid_strict.json`).

| block | model | notes |
|---|---|---|
| `rfclip` 768 | `openai/clip-vit-large-patch14` @32bd6428, `image_embeds` | fp32 |
| `rfibf` 1024 | ImageBind-huge vision, fp16 | Resize 224 bicubic + CenterCrop + CLIP mean/std |
| `rfsam` 256 | SimAM-ResNet100 VoxBlink2 ft (WeSpeaker) | first 60 s |
| `rfr221` 256 | WeSpeaker ResNet221-LM | whole clip |
| `rfxv` 192 | SpeechBrain `yangwang825/ecapa-tdnn-vox2` (BTC space) | whole clip |
| `rfibv` 1024 | ImageBind-huge audio, fp16 | 2 s clips covering the clip (>= 3) |
| `rfagef` 10 / `rfagev` 4 | `nateraw/vit-age-classifier`, `audeering/wav2vec2-large-robust-24-ft-age-gender` | only v4 train + dev lists |

Lane a = MAV-Celeb rows (FLAG_12 loop). Lanes b / c = two interleaved halves of the aux store (faces: odd / even frame rows; voices: the audio cache chunks).
Deliberate differences from RF: aux voices are the cached centre 3 s clips (not whole clips), aux faces the cached 224 PNG frames; MAV audio is decoded by the
FLAG_12 loader (librosa soxr resample; only MAV v1 is not 16 kHz) and v1 / v2 voices are the 12 s centre crop as every array of this store.
**Private** (VoxBlink rows). GPU T4 x 2, Internet ON.

In [ ]:
print("[boot] FLAG_162b_rfs_auxb | python", __import__("sys").version.split()[0], "| start", __import__("time").strftime("%Y-%m-%d %H:%M:%S"), flush=True)  # heartbeat rule: first output, before any import
print("[cell] b1", flush=True)
import os, sys, io, re, time, json, types, shutil, zipfile, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
LOCAL = os.environ.get("FLAG_LOCAL") == "1"          # CPU smoke test on a mini copy of the data
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORKDIR = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_cache"))           # checkpoints stay out of the output
ECAPA_DIR = Path(os.environ.get("FLAG_ECAPA", str(CACHE / "ecapa_sb")))
OUT = WORKDIR / "feats_rfs_b"
for d in (WORKDIR, CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

# ------------------------------------------------------------------ what to run (edit here)
ENCODERS = ['rfface']   # RFS-01 step 1 (replaced below)
EXT_SOURCES = ["v1_complete", "v2_complete"]                        # MAV-Celeb rows for STRESS-01; [] = skip
EXT_ENCODERS = list(ENCODERS)                                       # every model also on the v1/v2 rows
SPLIT_FILTER = None            # e.g. ["no_gender/English"]; None = every split found (train + every test list)
ZIP_PREFIXES = ("train_set", "dev_set", "eval", "test")             # organiser zips unpacked when not extracted
ONLY_UNDER = None              # e.g. "eval": keep only lists whose path contains this (dev and eval both attached)
print("LOCAL" if LOCAL else "KAGGLE", "| input", IN, "| output", OUT, "| cache", CACHE)
SMOKE_ROWS = int(os.environ.get("FLAG_SMOKE_ROWS", "0"))   # local smoke test only: rows kept per split / source
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and os.environ.get("FLAG_ENCODERS"):              # local smoke test only: a subset of the encoders
    ENCODERS = EXT_ENCODERS = os.environ["FLAG_ENCODERS"].split(",")
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and os.environ.get("FLAG_SPLITS"):                # local smoke test only: a subset of the organiser lists
    SPLIT_FILTER = os.environ["FLAG_SPLITS"].split(",")

In [ ]:
print("[cell] b2", flush=True)
# speechbrain -U: Kaggle's preinstalled one breaks on recent torchaudio. insightface / ImageBind --no-deps: their
# declared deps would replace onnxruntime-gpu by the CPU build and pin an old torch. ImageBind's data loaders need
# pytorchvideo, so the two loaders used here are reimplemented below and its module is stubbed.
if not LOCAL:
    !pip -q install -U "speechbrain>=1.1.1" onnxruntime-gpu onnx iopath ftfy omegaconf fvcore 2>&1 | tail -2
    !pip -q install --no-deps "insightface==2.0" "git+https://github.com/facebookresearch/ImageBind" 2>&1 | tail -2
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, soundfile as sf, transformers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# RF's own install recipe (ImageBind imports) on top of the framework's PIP12 cell
if not LOCAL:
    import subprocess, importlib.util
    for mod in ("wcwidth", "regex", "portalocker", "einops", "timm", "hyperpyyaml"):
        if importlib.util.find_spec(mod) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", mod], check=True)
    import importlib.metadata as md_
    if tuple(int(x) for x in md_.version("ruamel.yaml").split(".")[:2]) >= (0, 19):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ruamel.yaml<0.19"], check=True)

## 1. Find the data (FLAG_12 cell): organiser lists with media, v1/v2 rows, resume

In [ ]:
print("[cell] b3", flush=True)
# ============================================================================ finding the data
def scan_inputs(inp: Path, media=("faces", "voices")):
    """One os.walk over the inputs that never enters a faces/ or voices/ folder (an extracted MAV-Celeb holds hundreds of
    thousands of files; every rglob over it costs minutes). The organiser csv live in features/{faces,voices}/: entered.
    -> {"files": name -> [paths], "roots": folders holding both faces/ and voices/}"""
    files, roots = {}, []
    for dp, dns, fns in os.walk(inp, followlinks=True):
        feat = Path(dp).name == "features"
        if not feat and all(m in dns for m in media):
            roots.append(Path(dp))
        dns[:] = sorted(d for d in dns if feat or d not in media)
        for f in fns:
            files.setdefault(f, []).append(Path(dp) / f)
    log(f"scanned {inp}: {sum(map(len, files.values()))} files outside media folders, {len(roots)} media roots")
    return dict(files=files, roots=roots)


def media_cols(txt: Path):
    """Column of the .wav and of the image in an organiser list (train: 2 / 3; test: 1 / 2), or None."""
    try:
        row = pd.read_csv(txt, sep=" ", header=None, nrows=1).iloc[0]
    except Exception:
        return None
    wav = [i for i, v in enumerate(row) if str(v).lower().endswith(".wav")]
    jpg = [i for i, v in enumerate(row) if str(v).lower().endswith((".jpg", ".jpeg", ".png"))]
    if not wav or not jpg:
        return None
    return wav[0], jpg[0]


def has_media(txt: Path, cols):
    row = pd.read_csv(txt, sep=" ", header=None, nrows=1).iloc[0]
    return (txt.parent / row[cols[0]]).exists() and (txt.parent / row[cols[1]]).exists()


def split_name(txt: Path):
    """train_English.txt -> 'train'; <folder>/English_test.txt -> '<folder>/English' (dev: 'no_gender/English')."""
    if txt.name == "train_English.txt":
        return "train"
    return f"{txt.parent.name}/{re.sub(r'_test$', '', txt.stem)}"


tag = lambda split: split.replace("/", "_")


def find_splits(scan):
    """Every organiser list whose media sit beside it. A features-only copy of a list (no media) is ignored.
    Two different lists with the same split name (dev and Evaluation both attached) stop the run: set ONLY_UNDER."""
    S = {}
    for name, paths in scan["files"].items():
        if not name.endswith(".txt") or not (name == "train_English.txt" or name.endswith("_test.txt")):
            continue
        for p in sorted(paths, key=lambda q: len(q.as_posix())):
            if ONLY_UNDER and ONLY_UNDER not in p.as_posix() and name != "train_English.txt":
                continue
            cols = media_cols(p)
            if cols is None or not has_media(p, cols):
                continue
            k = split_name(p)
            df = pd.read_csv(p, sep=" ", header=None)
            if k in S:
                same = len(df) == len(S[k]["wav"]) and df[cols[0]].tolist() == S[k]["wav"]
                assert same, f"two different lists named {k}: {S[k]['txt']} and {p} -> set ONLY_UNDER"
                continue
            S[k] = dict(txt=p, root=p.parent, wav=df[cols[0]].tolist(), jpg=df[cols[1]].tolist())
    return S


def unpack_zips(scan, work: Path):
    zips = [p for n, ps in scan["files"].items() if n.lower().endswith(".zip") and n.lower().startswith(ZIP_PREFIXES)
            for p in ps]
    for zp in zips:
        marker = work / f".{zp.stem}.done"
        if not marker.exists():
            log("unpacking", zp)
            with zipfile.ZipFile(zp) as z:
                z.extractall(work)
            marker.touch()
    return bool(zips)


SCAN = scan_inputs(IN)
S = find_splits(SCAN)
if "train" not in S or len(S) == 1:                  # something may still be zipped (Kaggle keeps some uploads as .zip)
    if unpack_zips(SCAN, WORKDIR / "raw"):
        for k, v in find_splits(scan_inputs(WORKDIR / "raw")).items():
            S.setdefault(k, v)
if SPLIT_FILTER:
    S = {k: v for k, v in S.items() if k in SPLIT_FILTER}
assert S, "no organiser list with media found: attach the raw v4 data (or the Evaluation data)"
def wav_duration(p):
    """Seconds; 1.0 for a file soundfile cannot open (load_audio then returns 1 s of silence for it)."""
    try:
        return sf.info(str(p)).duration
    except Exception:
        return 1.0


for k, v in S.items():
    v["dur"] = np.array([wav_duration(v["root"] / p) for p in v["wav"]], dtype=np.float32)
    shutil.copy(v["txt"], OUT / f"index_{tag(k)}.txt")
    log(f"{k:22s} {len(v['wav']):5d} rows | voice median {np.median(v['dur']):.1f}s, max {v['dur'].max():.1f}s | {v['txt']}")
meta = SCAN["files"].get("meta_file_train_set.csv")
if meta:
    shutil.copy(meta[0], OUT / "meta_file_train_set.csv")


# ============================================================================ MAV-Celeb v1/v2 rows (optional)
def media_key(member):
    """'.../faces/<id>/<lang>/<video>/<file>' -> 'faces/<id>/<lang>/<video>/<file>' (drops any version folder)."""
    p = member.replace("\\", "/").split("/")
    for kind in ("faces", "voices"):
        if kind in p:
            return "/".join(p[p.index(kind):])
    return None


def ext_source(scan, name):
    """The attached zip for `name` (largest copy), else the extracted folder whose path names the source."""
    norm = lambda t: t.lower().replace("-", "_")
    zips = [p for n, ps in scan["files"].items() if n.lower().endswith(".zip") and norm(name) in norm(n) for p in ps]
    if zips:
        return max(zips, key=lambda p: p.stat().st_size)
    short = name.split("_")[0]
    for r in scan["roots"]:
        hint = norm(r.as_posix()) + "/"
        if f"/{short}/" in hint or name in hint:
            return r
    return None


class ExtRows:
    """The exact rows of ext_<name>_{face,voice}_meta.csv, read from the attached zip or folder (row order = csv)."""

    def __init__(self, src, name, meta_dir: Path, max_sec=12.0):
        fm = pd.read_csv(meta_dir / f"ext_{name}_face_meta.csv").path
        vm = pd.read_csv(meta_dir / f"ext_{name}_voice_meta.csv").path
        src = Path(src)
        if src.is_dir():
            from concurrent.futures import ThreadPoolExecutor
            self.face, self.voice = [src / media_key(p) for p in fm], [src / media_key(p) for p in vm]
            with ThreadPoolExecutor(32) as ex:
                ok = list(ex.map(lambda q: q.exists(), self.face + self.voice))
            if not all(ok):
                raise KeyError(f"{ok.count(False)} rows missing, e.g. {(self.face + self.voice)[ok.index(False)]}")
            self.read = lambda q: q.read_bytes()
        else:
            z = zipfile.ZipFile(src)
            by = {media_key(i.filename): i.filename for i in z.infolist() if not i.is_dir() and media_key(i.filename)}
            self.face, self.voice = [by[media_key(p)] for p in fm], [by[media_key(p)] for p in vm]
            self.read = z.read
        self.L = int(max_sec * 16000)          # 12 s centre crop, as the organiser-space features of these rows
        log(f"{name}: {len(self.face)} face rows, {len(self.voice)} voice rows in {src}")


EXT = {}
for name in EXT_SOURCES:
    metas = sorted(SCAN["files"].get(f"ext_{name}_voice_meta.csv", []), key=lambda p: -p.stat().st_size)
    src = ext_source(SCAN, name)
    if metas and src is not None:
        try:
            EXT[name] = ExtRows(src, name, metas[0].parent)
        except Exception as e:                    # optional: a bad zip or a missing row must not stop the run
            log(f"{name}: skipped ({type(e).__name__}: {e})")
    else:
        log(f"{name}: skipped (meta csv found: {bool(metas)}, source found: {src is not None})")


# ============================================================================ resume
def seed_from_previous(scan, out: Path):
    """Copy arrays of an earlier run of THIS notebook (a folder named like flag2027-feats-eval / feats_eval), so finished
    work is skipped. Arrays of other datasets (feats_v2, feats_models) are references for the check, never copied."""
    copied = []
    for n, ps in scan["files"].items():
        if not n.endswith(".npy") or (out / n).exists():
            continue
        for p in ps:
            if re.search(r"feats[-_]rfs[-_]b", p.as_posix()) and out not in p.parents:
                shutil.copy(p, out / n); copied.append(n); break
    log(f"resume: {len(copied)} arrays copied from an earlier run")
    return copied


SEEDED = seed_from_previous(SCAN, OUT)
print({k: len(v["wav"]) for k, v in S.items()}, "| external:", list(EXT))
if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") and SMOKE_ROWS:                     # local smoke test: the first rows of every split and source
    for v in S.values():
        v.update(wav=v["wav"][:SMOKE_ROWS], jpg=v["jpg"][:SMOKE_ROWS], dur=v["dur"][:SMOKE_ROWS])
    for r in EXT.values():
        r.face, r.voice = r.face[:SMOKE_ROWS], r.voice[:SMOKE_ROWS]
    log(f"smoke test: every split and source cut to {SMOKE_ROWS} rows")

## 2. Audio loading, batching, helpers (FLAG_12 cell)

In [ ]:
print("[cell] b4", flush=True)
# ============================================================================ audio + batching
TARGET_SR = 16000
AUDIO_STATS = {"files": 0, "resampled": 0, "multichannel": 0, "short": 0}
MIN_SEC = 0.1                  # shorter clips are tiled to 0.1 s (w2v2 / AudioCLIP STFT fail on a few ms; v4 clips are >= 2 s)
BROKEN = {"audio": [], "image": []}   # unreadable sources: scored from a stand-in, listed in the manifest


def _src_name(src):
    """Full path of a file source (base names repeat across splits); the archive member name of a zip stream."""
    if isinstance(src, (bytes, bytearray)):
        return f"<{len(src)} bytes>"
    return str(src) if isinstance(src, (str, Path)) else str(getattr(src, "name", None) or src)


def load_audio(src):
    """Mono float32 at 16 kHz as the organiser loads audio (librosa.load(sr=16000, mono=True): channel mean + soxr_hq).
    All v4 wavs are 16 kHz mono; MAV-Celeb v1 has 44.1 kHz files. `src` = path or file-like.
    A file that cannot be read (or is empty / non-finite) -> 1 s of silence, listed in BROKEN["audio"]."""
    try:
        w, sr = sf.read(src if hasattr(src, "read") else str(src), dtype="float32", always_2d=True)
        if w.size == 0 or not np.isfinite(w).all():
            raise ValueError(f"{w.shape[0]} samples, finite {bool(np.isfinite(w).all())}")
    except Exception as e:
        BROKEN["audio"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable audio -> 1 s of silence:", BROKEN["audio"][-1])
        return np.zeros(TARGET_SR, np.float32)
    AUDIO_STATS["files"] += 1
    if w.shape[1] > 1:
        AUDIO_STATS["multichannel"] += 1
        w = w.mean(1, keepdims=True)
    w = w[:, 0]
    if sr != TARGET_SR:
        import librosa
        w = librosa.resample(w, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
        AUDIO_STATS["resampled"] += 1
    if len(w) < int(MIN_SEC * TARGET_SR):
        AUDIO_STATS["short"] += 1
        w = np.tile(w, int(np.ceil(MIN_SEC * TARGET_SR / len(w))))[:int(MIN_SEC * TARGET_SR)]
    return w


def centre_crop(w, L):
    if len(w) <= L:
        return w
    s = (len(w) - L) // 2
    return w[s:s + L]


def pack_batches(durs, max_padded_sec=240.0, max_bs=32):
    """Longest first; a batch costs len(batch) * longest item after padding."""
    order = np.argsort(-np.asarray(durs), kind="stable")
    batches, cur = [], []
    for i in order:
        longest = durs[cur[0]] if cur else durs[i]
        if cur and ((len(cur) + 1) * longest > max_padded_sec or len(cur) >= max_bs):
            batches.append(cur); cur = []
        cur.append(int(i))
    if cur:
        batches.append(cur)
    return batches


def run_batched(items, fn, durs, max_padded_sec=240.0, max_bs=32, cpu_fn=None, label=""):
    """fn(list of items) -> list of outputs, in item order. A CUDA OOM halves the batch; a single item goes to CPU."""
    out = [None] * len(items)
    t0, done_n = time.time(), 0

    def go(idx):
        try:
            res = fn([items[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) == 1:
                res = (cpu_fn or fn)([items[idx[0]]])
            else:
                h = len(idx) // 2
                go(idx[:h]); go(idx[h:])
                return
        for i, r in zip(idx, res):
            out[i] = r

    batches = pack_batches(durs, max_padded_sec, max_bs)
    for bi, b in enumerate(batches):
        go(b)
        done_n += len(b)
        if bi % 50 == 0 or done_n == len(items):
            el = time.time() - t0
            log(f"  {label} {done_n}/{len(items)}  {el:.0f}s  eta {el / done_n * (len(items) - done_n):.0f}s")
    return out


def cos_rows(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    return np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)


def amp(flag):
    return torch.autocast("cuda", dtype=torch.float16, enabled=bool(flag) and DEV.type == "cuda")


def read_rgb(src):
    """PIL RGB image; an image that cannot be read -> a black 224 x 224 image, listed in BROKEN["image"]."""
    from PIL import Image
    try:
        return Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src).convert("RGB")
    except Exception as e:
        BROKEN["image"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable image -> black image:", BROKEN["image"][-1])
        return Image.new("RGB", (224, 224))

## 3. Encoders

In [ ]:
print("[cell] b5", flush=True)
# ============================================================================ encoders
# Face encoders: embed(list of sources) -> {suffix: (N, d)}; "" is the main array. A source is a path or raw bytes.
# Voice encoders: embed(list of waveforms, durations) -> {suffix: (N, d)}.

class ArcFace:
    """insightface buffalo_l: SCRFD detection + 5-point norm_crop + ArcFace R100 (512). fp32, as FLAG_04."""
    kind = "face"

    def __init__(self):
        import onnxruntime as ort
        from insightface.app import FaceAnalysis
        prov = [p for p in ["CUDAExecutionProvider", "CPUExecutionProvider"] if p in ort.get_available_providers()]
        try:
            self.app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "recognition"], providers=prov)
        except TypeError:
            self.app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection", "recognition"])
        self.app.prepare(ctx_id=0 if DEV.type == "cuda" else -1, det_size=(224, 224), det_thresh=0.3)
        self.rec = self.app.models["recognition"]
        log("ArcFace recognition runs on", self.rec.session.get_providers())

    def align(self, img):
        """BGR image -> (112 x 112 crop on the ArcFace 5-point template, detected?). Keeps the big central face."""
        import cv2
        from insightface.utils import face_align
        faces = self.app.det_model.detect(img, max_num=0, metric="default")
        bboxes, kpss = faces if isinstance(faces, tuple) else (faces, None)
        if bboxes is not None and len(bboxes) and kpss is not None:
            h, w = img.shape[:2]
            ctr = np.array([w / 2, h / 2])
            area = (bboxes[:, 2] - bboxes[:, 0]) * (bboxes[:, 3] - bboxes[:, 1])
            dist = np.linalg.norm((bboxes[:, :2] + bboxes[:, 2:4]) / 2 - ctr, axis=1)
            j = int(np.argmax(area - 2.0 * dist ** 2))
            return face_align.norm_crop(img, landmark=kpss[j], image_size=112), True
        h, w = img.shape[:2]; m = int(min(h, w) * 0.15)
        return cv2.resize(img[m:h - m, m:w - m], (112, 112)), False

    def embed(self, srcs, label=""):
        import cv2
        E, OK, t0 = [], [], time.time()
        for i, s in enumerate(srcs):
            img = cv2.imdecode(np.frombuffer(s, np.uint8), cv2.IMREAD_COLOR) if isinstance(s, (bytes, bytearray)) \
                else cv2.imread(str(s))
            if img is None:                                # unreadable: black image (the pair is still scored)
                BROKEN["image"].append(f"{_src_name(s)} | cv2 could not decode it")
                log("!! unreadable image -> black image:", BROKEN["image"][-1])
                img = np.zeros((224, 224, 3), np.uint8)
            a, ok = self.align(img)
            E.append(self.rec.get_feat(a).flatten().astype(np.float32)); OK.append(ok)
            if i % 2000 == 0:
                log(f"  arcface {label} {i}/{len(srcs)} {time.time() - t0:.0f}s")
        log(f"  arcface {label}: detection + alignment {100 * np.mean(OK):.1f}%")
        return {"": np.stack(E), "_detok": np.array(OK)}


def speechbrain_import():
    import torchaudio
    for n, f in [("list_audio_backends", lambda: ["soundfile"]), ("get_audio_backend", lambda: "soundfile"),
                 ("set_audio_backend", lambda *a, **k: None)]:
        if not hasattr(torchaudio, n):
            setattr(torchaudio, n, f)
    from speechbrain.inference.speaker import EncoderClassifier
    return EncoderClassifier


class Ecapa:
    """speechbrain/spkrec-ecapa-voxceleb -> 192 (ecapa192) + attentive-statistics pooling 6144 (ecapa6144). fp32."""
    kind = "voice"

    def __init__(self):
        EncoderClassifier = speechbrain_import()
        kw = {}
        try:
            from speechbrain.utils.fetching import LocalStrategy
            kw["local_strategy"] = LocalStrategy.COPY
        except Exception:
            pass
        self.m = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir=str(ECAPA_DIR),
                                                run_opts={"device": str(DEV)}, **kw)
        self.m.eval()
        self._pool = {}
        self.m.mods.embedding_model.asp.register_forward_hook(lambda mod, i, o: self._pool.__setitem__("x", o))

    @torch.no_grad()
    def batch(self, sigs, device=None):
        device = device or DEV
        lens = [len(s) for s in sigs]
        L = max(lens)
        x = torch.zeros(len(sigs), L)
        for i, s in enumerate(sigs):
            x[i, :len(s)] = torch.from_numpy(s)
        rel = torch.tensor([l / L for l in lens], dtype=torch.float32, device=device)
        mods = self.m.mods if device == DEV else self._cpu_mods()
        f = mods.mean_var_norm(mods.compute_features(x.to(device)), rel)
        e = mods.embedding_model(f, rel).squeeze(1)
        p = self._pool["x"].squeeze(-1)
        return list(zip(e.float().cpu().numpy(), p.float().cpu().numpy()))

    def _cpu_mods(self):
        if not hasattr(self, "_cpu"):
            import copy
            self._cpu = copy.deepcopy(self.m.mods).to("cpu")
            self._cpu.embedding_model.asp.register_forward_hook(lambda mod, i, o: self._pool.__setitem__("x", o))
        return self._cpu

    def embed(self, load, n, durs, label=""):
        res = run_batched(list(range(n)), lambda ii: self.batch([load(i) for i in ii]), durs,
                          cpu_fn=lambda ii: self.batch([load(i) for i in ii], device="cpu"), label=f"ecapa {label}")
        return {"192": np.stack([r[0] for r in res]).astype(np.float32),
                "6144": np.stack([r[1] for r in res]).astype(np.float32)}


class ViTAge:
    """nateraw/vit-age-classifier: post-LayerNorm CLS (768) + the 9 FairFace age-bin probabilities (fp16 autocast)."""
    kind = "face"
    ID = "nateraw/vit-age-classifier"

    def __init__(self):
        from transformers import ViTImageProcessor, ViTForImageClassification
        self.proc = ViTImageProcessor.from_pretrained(self.ID)
        self.m = ViTForImageClassification.from_pretrained(self.ID).to(DEV).eval()

    def prep(self, im):
        return self.proc(images=im, return_tensors="pt").pixel_values[0]

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        with amp(use_amp):
            h = self.m.vit(x.to(DEV)).last_hidden_state[:, 0]
            p = torch.softmax(self.m.classifier(h).float(), 1)
        return {"": h.float().cpu().numpy(), "prob": p.cpu().numpy()}


class AgeGenderHead(nn.Module):
    def __init__(self, config, num_labels):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, num_labels)

    def forward(self, x):
        return self.out_proj(self.dropout(torch.tanh(self.dense(self.dropout(x)))))


class W2V2AgeGender:
    """audeering/wav2vec2-large-robust-24-ft-age-gender: mean hidden state (1024), age (x 100 = years), gender
    probabilities (female, male, child). 20 s windows, 10 s hop, length-weighted mean. fp16 autocast."""
    kind = "voice"
    ID = "audeering/wav2vec2-large-robust-24-ft-age-gender"

    def __init__(self):
        from transformers import Wav2Vec2Processor
        from transformers.models.wav2vec2.modeling_wav2vec2 import Wav2Vec2Model, Wav2Vec2PreTrainedModel

        class AgeGenderModel(Wav2Vec2PreTrainedModel):
            def __init__(self, config):
                super().__init__(config)
                self.config = config
                self.wav2vec2 = Wav2Vec2Model(config)
                self.age = AgeGenderHead(config, 1)
                self.gender = AgeGenderHead(config, 3)
                self.post_init()        # not init_weights(): transformers >= 5 sets all_tied_weights_keys here

        # transformers 5.0 reads sys.modules[cls.__module__].__file__; a class defined in a notebook lives in __main__,
        # which has no __file__ under Jupyter (w2v2ag failed on Kaggle). Point it at the real wav2vec2 module.
        AgeGenderModel.__module__ = Wav2Vec2Model.__module__
        self.proc = Wav2Vec2Processor.from_pretrained(self.ID)
        self.m = AgeGenderModel.from_pretrained(self.ID).to(DEV).eval()
        self.W, self.H = 20 * 16000, 10 * 16000

    def windows(self, w):
        starts = [0] if len(w) <= self.W else list(range(0, len(w) - self.W + self.H, self.H))
        for i, s in enumerate(starts):
            seg = w[s:s + self.W]
            if i and len(seg) < 16000:
                break
            yield seg

    @torch.no_grad()
    def __call__(self, w, use_amp=True):
        acc, tot = None, 0
        for seg in self.windows(w):
            x = self.proc(seg, sampling_rate=16000, return_tensors="pt").input_values.to(DEV)
            with amp(use_amp):
                h = self.m.wav2vec2(x)[0].float().mean(1)[0]
            acc = h * len(seg) if acc is None else acc + h * len(seg)
            tot += len(seg)
        h = (acc / tot)[None]
        return {"": h[0].cpu().numpy(), "age": self.m.age(h)[0].cpu().numpy(),
                "gen": torch.softmax(self.m.gender(h), 1)[0].cpu().numpy()}


IB_URL = "https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth"
_IB = {}


def imagebind():
    """ImageBind-huge, loaded once for vision and audio. imagebind/__init__ imports imagebind.data (needs
    pytorchvideo): stubbed, the two loaders used are reimplemented below."""
    if "m" not in _IB:
        sys.modules.setdefault("imagebind.data", types.ModuleType("imagebind.data"))
        from imagebind.models import imagebind_model
        from imagebind.models.imagebind_model import ModalityType
        f = CACHE / "imagebind_huge.pth"
        if not f.exists():
            log("downloading", IB_URL)
            torch.hub.download_url_to_file(IB_URL, str(f) + ".part", progress=False)
            Path(str(f) + ".part").rename(f)
        m = imagebind_model.imagebind_huge(pretrained=False)
        m.load_state_dict(torch.load(f, map_location="cpu", mmap=True))
        _IB.update(m=m.to(DEV).eval(), MT=ModalityType)
    return _IB


class IBVision:
    kind = "face"

    def __init__(self):
        from torchvision import transforms as T
        self.ib = imagebind()
        self.tf = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(224), T.ToTensor(),
                             T.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711))])

    def prep(self, im):
        return self.tf(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        with amp(use_amp):
            e = self.ib["m"]({self.ib["MT"].VISION: x.to(DEV)})[self.ib["MT"].VISION]
        return {"": e.float().cpu().numpy()}


def ib_audio_clips(w, sr=16000, clip=2.0, mel=128, frames=204, mean=-4.268, std=9.138):
    """imagebind.data.load_and_transform_audio_data for one waveform, with n = max(3, ceil(duration / 2 s)) clips evenly
    spaced from start to end (covers the whole file; ImageBind's default n = 3 is identical up to 6 s)."""
    import torchaudio
    n = max(3, int(np.ceil(len(w) / sr / clip)))
    mx = max(len(w) / sr - clip, 0.0)
    out = []
    for k in range(n):
        s = mx * k / max(n - 1, 1)
        seg = torch.from_numpy(np.ascontiguousarray(w[int(s * sr):int((s + clip) * sr)], dtype=np.float32))[None]
        if seg.shape[1] < 400:
            seg = F.pad(seg, (0, 400 - seg.shape[1]))
        seg = seg - seg.mean()
        fb = torchaudio.compliance.kaldi.fbank(seg, htk_compat=True, sample_frequency=sr, use_energy=False,
                                               window_type="hanning", num_mel_bins=mel, dither=0.0,
                                               frame_length=25, frame_shift=10).T
        fb = F.pad(fb, (0, frames - fb.shape[1])) if fb.shape[1] < frames else fb[:, :frames]
        out.append((fb[None] - mean) / std)
    return torch.stack(out)


class IBAudio:
    kind = "voice"

    def __init__(self):
        self.ib = imagebind()

    @torch.no_grad()
    def __call__(self, w, use_amp=True):
        x = ib_audio_clips(w)[None].to(DEV)                       # the model averages the clips
        with amp(use_amp):
            e = self.ib["m"]({self.ib["MT"].AUDIO: x})[self.ib["MT"].AUDIO]
        return {"": e[0].float().cpu().numpy()}


def build(name):
    return {"arcface": ArcFace, "ecapa": Ecapa, "vitage": ViTAge, "w2v2ag": W2V2AgeGender,
            "ibv": IBVision, "iba": IBAudio}[name]()


def release(keep_ib=False):
    if not keep_ib:
        _IB.clear()
    import gc
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
print("[cell] b6", flush=True)
# ============================================================================ RFS-01 step 1: RF's extractors (core embedded verbatim) and checkpoints
import base64, zlib, hashlib, importlib.util, urllib.request
CORE_B64 = "..."
CORE_SHA = "25abe24db6fbe863109dafd5f75f9075164dce75f77ce276695daf8687d338f7"
RFS_DIR = WORKDIR / "rfs_code"; RFS_DIR.mkdir(exist_ok=True)
_src = zlib.decompress(base64.b64decode(CORE_B64))
assert hashlib.sha256(_src).hexdigest() == CORE_SHA, "embedded RF core differs from the registered file"
(RFS_DIR / "flag_final_core_092.py").write_bytes(_src)
log("RF core sha256", CORE_SHA[:12])


def load_core(dev, tag):
    """One module object per device (the core reads its global DEVICE at call time): a lane may run faces on cuda:0 and voices on cuda:1."""
    spec = importlib.util.spec_from_file_location(f"flag_final_core_092_{tag}", RFS_DIR / "flag_final_core_092.py")
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); m.DEVICE = torch.device(dev)
    return m


URL = {"imagebind": "https://dl.fbaipublicfiles.com/imagebind/imagebind_huge.pth",
       "r221": "https://huggingface.co/Wespeaker/wespeaker-voxceleb-resnet221-LM/resolve/main/voxceleb_resnet221_LM.pt",
       "sam": "https://huggingface.co/MCplayer/voxblink2_samresnet100_ft/resolve/main/models/voxblink2_samresnet100_ft/avg_model.pt"}
SHA = {"imagebind": "d6f6c22bedcc90708448d5d2fbb7b2db9c73f505dc89bd0b2e09b23af1b62157",
       "r221": "47d76239f1e865b273e31e30f691959061ff6565ed6c4e7be9639a65d9662eb5",
       "sam": "5aeee438ca23c0ca6e341bab6c6bf7f465497e1dc323bb1bc1074d6a0c778b11"}
CK = {}
for _n, _u in URL.items():
    _t = CACHE / "rfs_models" / f"{_n}.pt"; _t.parent.mkdir(exist_ok=True)
    for _k in range(4):
        if _t.exists():
            break
        try:
            log("download", _n); urllib.request.urlretrieve(_u, str(_t) + ".part"); Path(str(_t) + ".part").replace(_t)
        except OSError as e:
            log("  download error", _n, type(e).__name__, "- retry"); Path(str(_t) + ".part").unlink(missing_ok=True); time.sleep(30 * (_k + 1))
    _h = hashlib.sha256()
    with open(_t, "rb") as f_:
        for b_ in iter(lambda: f_.read(1 << 24), b""):
            _h.update(b_)
    assert _h.hexdigest() == SHA[_n], f"{_n}: sha256 differs from the registered checkpoint"
    CK[_n] = _t
log("checkpoints", {k: str(v.stat().st_size // 2 ** 20) + " MB" for k, v in CK.items()})
from PIL import Image
import torchvision.transforms as TV


def tile1s(w):
    """load_wave's rule: a clip shorter than 1 s is tiled to 1 s (uses only this clip)."""
    return w if len(w) >= 16000 else np.tile(w, int(np.ceil(16000 / len(w))))[:16000]


class RFFace:
    """CLIP-L image_embeds (fp32) and ImageBind vision (fp16 on GPU) of one PIL image each, exactly ClipFaceExtractor.faces / L1Extractor.faces."""

    def __init__(self, c, dev):
        self.c, self.dev = c, torch.device(dev); self.half = self.dev.type == "cuda"
        self.clip = c.ClipFaceExtractor()
        self.ib, self.MT = c.load_imagebind(CK["imagebind"])
        if not self.half:
            self.ib = self.ib.float()
        self.vis = TV.Compose([TV.Resize(224, interpolation=TV.InterpolationMode.BICUBIC), TV.CenterCrop(224), TV.ToTensor(),
                               TV.Normalize(c.IB_VIS_MEAN, c.IB_VIS_STD)])

    def prep(self, im):
        im = im.convert("RGB")
        return torch.stack([self.clip.p(images=im, return_tensors="pt")["pixel_values"][0], self.vis(im)])

    @torch.inference_mode()
    def run(self, x):
        cl = self.clip.m(pixel_values=x[:, 0].to(self.dev)).image_embeds.float().cpu().numpy()
        v = x[:, 1].to(self.dev)
        ibf = self.ib({self.MT.VISION: v.half() if self.half else v})[self.MT.VISION].float().cpu().numpy()
        return {"": ibf, "clip": cl}


class RFVoice:
    """ImageBind audio, ResNet221-LM, SimAM-ResNet100 (60 s), SpeechBrain ECAPA of one 16 kHz waveform (L2Extractor.voice_features, VB2Extractor)."""

    def __init__(self, c, dev):
        self.c, self.dev = c, torch.device(dev); self.half = self.dev.type == "cuda"
        self.ib, self.MT = c.load_imagebind(CK["imagebind"])
        if not self.half:
            self.ib = self.ib.float()
        r = c.ENCODERS["resnet221_lm"]["make"]()
        sd = torch.load(CK["r221"], map_location="cpu"); sd = sd.get("model", sd)
        res = r.load_state_dict(sd, strict=False)
        if res.missing_keys or set(res.unexpected_keys) - {"projection.weight"}:
            raise ValueError(f"ResNet221 checkpoint mismatch: {res}")
        self.r221 = r.to(self.dev).eval()
        self.sam = c.VB2Extractor(CK["sam"]).m
        EncoderClassifier = speechbrain_import()
        from speechbrain.utils.fetching import LocalStrategy
        self.btc = EncoderClassifier.from_hparams(source="yangwang825/ecapa-tdnn-vox2", savedir=str(CACHE / "sb_btc"),
                                                  run_opts={"device": str(self.dev)}, local_strategy=LocalStrategy.COPY)

    @torch.inference_mode()
    def feats(self, w):
        c = self.c; w = tile1s(np.ascontiguousarray(w, dtype=np.float32))
        a = c.ib_audio_clips(w)[None].to(self.dev)
        ibv = self.ib({self.MT.AUDIO: a.half() if self.half else a})[self.MT.AUDIO].float().cpu().numpy()[0]
        o = self.r221(c.kaldi_fbank(w)[None].to(self.dev))
        r2 = (o[-1] if isinstance(o, tuple) else o).squeeze(0).float().cpu().numpy()
        sam = self.sam(c.kaldi_fbank(w[:16000 * 60])[None].to(self.dev)).squeeze(0).float().cpu().numpy()
        xv = self.btc.encode_batch(torch.tensor(w)[None]).squeeze().float().cpu().numpy()
        return {"": ibv, "r221": r2, "sam": sam, "xv": xv}


class RFAge:
    """AgeExtractor's two models on one image / one waveform (face: [expected age, 9 probabilities]; voice: [age, p(f), p(m), p(child)])."""

    def __init__(self, c, dev):
        self.c, self.dev = c, torch.device(dev); self.ag = c.AgeExtractor()

    def prep(self, im):
        return self.ag.fp(im.convert("RGB"), return_tensors="pt")["pixel_values"][0]

    @torch.inference_mode()
    def face(self, x):
        pr = torch.softmax(self.ag.fm(x.to(self.dev)).logits, -1).float().cpu().numpy()
        return {"": np.concatenate([(pr @ self.c.AGE_MID)[:, None], pr], 1)}

    @torch.inference_mode()
    def voice(self, w):
        w = tile1s(np.ascontiguousarray(w, dtype=np.float32))
        x = self.ag.vp(w, sampling_rate=16000, return_tensors="pt")["input_values"].to(self.dev)
        a, g = self.ag.vm(x)
        return {"": np.r_[100 * a.float().cpu().numpy()[0], g.float().cpu().numpy()[0]].astype(np.float32)}

In [ ]:
print("[cell] b7", flush=True)
# ============================================================================ aux lanes: the MAV loop of this framework does nothing
ENCODERS = EXT_ENCODERS = []
NEW, ARRAYS_NEW, MAIN_NEW, RESHAPE = {}, {}, {}, {}


def build(name):
    raise RuntimeError(name)


def release(keep_ib=False):
    import gc
    gc.collect()

## 4. Extract every encoder over every split, then the v1/v2 rows (FLAG_12 loop, resumable)

In [ ]:
print("[cell] b8", flush=True)
# ============================================================================ running
def run_faces_tensor(enc, srcs, bs=64, label=""):
    out, t0 = {}, time.time()
    for s in range(0, len(srcs), bs):
        x = torch.stack([enc.prep(read_rgb(p() if callable(p) else p)) for p in srcs[s:s + bs]])
        for k, v in enc(x).items():
            out.setdefault(k, []).append(v)
        if (s // bs) % 40 == 0:
            log(f"  {label} {s + len(x)}/{len(srcs)}  {time.time() - t0:.0f}s")
    return {k: np.concatenate(v).astype(np.float32) for k, v in out.items()}


def run_voices_single(enc, n, load, label=""):
    """One file at a time: no padding enters a mean."""
    out, t0 = {}, time.time()
    for i in range(n):
        for k, v in enc(load(i)).items():
            out.setdefault(k, []).append(np.atleast_1d(v))
        if i % 1000 == 0 or i == n - 1:
            el = time.time() - t0
            log(f"  {label} {i + 1}/{n}  {el:.0f}s  eta {el / (i + 1) * (n - i - 1):.0f}s")
    return {k: np.stack(v).astype(np.float32) for k, v in out.items()}


# output names: {kind}_{array}_{split}.npy for organiser splits, ext_{src}_{kind}_{array}.npy for v1/v2
ARRAYS = {"arcface": ("face", {"": "arcface", "_detok": "arcface_detok"}),
          "ecapa": ("voice", {"192": "ecapa192", "6144": "ecapa6144"}),
          "vitage": ("face", {"": "vitage", "prob": "vitageprob"}),
          "w2v2ag": ("voice", {"": "w2v2ag", "age": "w2v2agage", "gen": "w2v2aggen"}),
          "ibv": ("face", {"": "ibv"}), "iba": ("voice", {"": "iba"})}
MAIN = {"arcface": "", "ecapa": "192", "vitage": "", "w2v2ag": "", "ibv": "", "iba": ""}
ARRAYS.update(ARRAYS_NEW); MAIN.update(MAIN_NEW)                         # this notebook's arrays


def _dt(v, split):
    """float16 for the v1/v2 rows (as FLAG_12) and for every wide array (layers / tokens); float32 otherwise."""
    return v if (split and v.ndim == 2 and v.shape[1] <= 4096) or v.ndim == 1 else v.astype(np.float16)


def target(name, suffix, split=None, src=None):
    kind, names = ARRAYS[name]
    return OUT / (f"{kind}_{names[suffix]}_{tag(split)}.npy" if split else f"ext_{src}_{kind}_{names[suffix]}.npy")


def complete(name, split=None, src=None):
    return all(target(name, s, split, src).exists() for s in ARRAYS[name][1])


def save(name, R, split=None, src=None):
    for s, v in R.items():                         # main array last: its presence marks the job as done
        if s != MAIN[name]:
            np.save(target(name, s, split, src), _dt(v, split))
    np.save(target(name, MAIN[name], split, src), _dt(R[MAIN[name]], split))


def extract(name, enc, split=None, src=None):
    kind = ARRAYS[name][0]
    if split:
        sp = S[split]
        faces = [sp["root"] / p for p in sp["jpg"]]
        load = lambda i: load_audio(sp["root"] / sp["wav"][i])
        n, durs, label = len(sp["wav"]), sp["dur"], split
    else:
        rows = EXT[src]
        faces = [rows.read(p) for p in rows.face] if name == "arcface" else [(lambda p=p: rows.read(p)) for p in rows.face]
        load = lambda i: centre_crop(load_audio(io.BytesIO(rows.read(rows.voice[i]))), rows.L)
        n, label = len(rows.voice), src
        durs = np.full(n, 12.0, np.float32)          # upper bound after the crop: batches stay within budget
    if kind == "face":
        R = enc.embed(faces, label=label) if name == "arcface" else run_faces_tensor(enc, faces, label=f"{name} {label}")
    elif name == "ecapa":
        R = enc.embed(load, n, durs, label=label)
    else:
        R = run_voices_single(enc, n, load, label=f"{name} {label}")
    save(name, R, split, src)
    log(f"saved {target(name, MAIN[name], split, src).name} {R[MAIN[name]].shape}")


def self_check(name, enc):
    """Batching / fp16 must not change the embeddings (min cosine over a few rows; must be > 0.995)."""
    k = next(k for k in S if k != "train") if len(S) > 1 else next(iter(S))
    sp = S[k]
    if name == "arcface":
        return {}
    if name == "ecapa":
        idx = list(np.argsort(-sp["dur"])[:3]) + list(range(min(5, len(sp["wav"]))))
        sigs = [load_audio(sp["root"] / sp["wav"][i]) for i in idx]
        b, s = enc.batch(sigs), [enc.batch([x])[0] for x in sigs]
        return {"batch192": float(cos_rows([r[0] for r in b], [r[0] for r in s]).min()),
                "batch6144": float(cos_rows([r[1] for r in b], [r[1] for r in s]).min())}
    if ARRAYS[name][0] == "face":
        x = torch.stack([enc.prep(read_rgb(sp["root"] / p)) for p in sp["jpg"][:6]])
        full = enc(x, use_amp=False)[""]
        single = np.concatenate([enc(x[i:i + 1], use_amp=False)[""] for i in range(len(x))])
        return {"batch": float(cos_rows(full, single).min()), "fp16": float(cos_rows(full, enc(x)[""]).min())}
    waves = [load_audio(sp["root"] / p) for p in sp["wav"][:3]]
    return {"fp16": float(cos_rows([enc(w, use_amp=False)[""] for w in waves], [enc(w)[""] for w in waves]).min())}


CHECKS, FAILED = {}, {}
for name in ENCODERS:
    jobs = [("split", k) for k in S if not complete(name, split=k)]
    if name in EXT_ENCODERS:
        jobs += [("src", s) for s in EXT if not complete(name, src=s)]
    if not jobs:
        log(f"== {name}: every array already exists -> skipped")
        continue
    t0 = time.time()
    try:
        enc = build(name)
        CHECKS[name] = self_check(name, enc)
        log(name, "self-check:", CHECKS[name])
        assert not CHECKS[name] or min(CHECKS[name].values()) > 0.995, f"{name}: batching / fp16 changes the output"
        for what, key in jobs:
            try:
                extract(name, enc, **({"split": key} if what == "split" else {"src": key}))
            except Exception as e:                  # one bad source must not cost the other splits
                FAILED[f"{name}:{key}"] = f"{type(e).__name__}: {e}"
                log(f"!! {name} on {key} FAILED: {FAILED[f'{name}:{key}']}")
    except Exception as e:
        FAILED[name] = f"{type(e).__name__}: {e}"
        log(f"!! {name} FAILED -> skipped: {FAILED[name]}")
    enc = None
    release(keep_ib=(name in ('ibv',)))
    log(f"== {name} done in {time.time() - t0:.0f}s")
print("failed:", FAILED or "none")

## 5. Zero-shot probe on true labels (v4 train, MAV-Celeb cross-video; never the dev lists)

In [ ]:
print("[cell] rfs check vs RF embeddings", flush=True)
# registered check: the RF-092 reproduction run (onnhqunhh/flag-rf-092-tailap-p, FLAG_RF_E2E/embeddings.npz, copied to the private dataset ngnnguynchn/flag2027-rf092-dev-emb-p: sorted unique dev files) vs the arrays above on
# the dev lists. Path-independent: every dev row is matched to its best row in the RF array on rfibf (faces) / rfibv (voices); then EVERY block (and the age) of the
# matched pair must agree (cos >= 0.999). Skipped without the reference attached.
_ref = SCAN["files"].get("embeddings.npz", [])
RFCHK = {}
if not _ref or 'auxb' != "mav":
    log("RF embeddings.npz not attached / aux lane: check skipped", bool(_ref))
else:
    Rz = np.load(_ref[0])
    def _cos(a, b):
        a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
        return np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)
    def _best(M, Rm):
        Mn = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-12); Rn = Rm / (np.linalg.norm(Rm, axis=1, keepdims=True) + 1e-12)
        sim = Mn @ Rn.T; j = sim.argmax(1)
        return j, sim[np.arange(len(M)), j]
    for kind, key, blocks in (("face", "ibf", [("rfclip", "clip"), ("rfibf", "ibf")]), ("voice", "ibv", [("rfsam", "sam"), ("rfr221", "r221"), ("rfxv", "xv"), ("rfibv", "ibv")])):
        Ms = {b: [] for b, _ in blocks}; Age = []; Rj = []
        for k in S:
            if k == "train":
                continue
            f = {b: OUT / f"{kind}_{b}_{tag(k)}.npy" for b, _ in blocks}
            if not all(p.exists() for p in f.values()):
                continue
            for b, _ in blocks:
                Ms[b].append(np.load(f[b]).astype(np.float32))
            fa = OUT / f"{kind}_rfage{kind[0]}_{tag(k)}.npy"
            Age.append(np.load(fa)[:, 0] if fa.exists() else np.full(len(Ms[blocks[0][0]][-1]), np.nan, np.float32))
        if not Ms[blocks[0][0]]:
            continue
        Ms = {b: np.concatenate(v) for b, v in Ms.items()}; Age = np.concatenate(Age)
        j, c_main = _best(Ms["rf" + key], Rz[f"{kind}_{key}"].astype(np.float32))
        res = {"rows": len(j), "match_cos_min": round(float(c_main.min()), 5)}
        for b, k_ in blocks:
            c = _cos(Ms[b], Rz[f"{kind}_{k_}"].astype(np.float32)[j]); res[b] = dict(cos_min=round(float(c.min()), 5), cos_mean=round(float(c.mean()), 6), below_0999=int((c < 0.999).sum()))
        if np.isfinite(Age).any():
            d = np.abs(Age - Rz[f"{kind}_age"][j]); res["age_max_abs_diff"] = round(float(np.nanmax(d)), 4)
        RFCHK[kind] = res
        log("RF check", kind, res)
    CHECKS["rfs01_vs_rf"] = RFCHK
    bad = {k: v for k, r in RFCHK.items() for k, v in r.items() if isinstance(v, dict) and v["cos_min"] < (0.99 if (os.environ.get("FLAG_SMOKE") == "1") else 0.999)}
    assert not bad, f"definition differs from RF: {bad}"

## 6. Manifest

In [ ]:
print("[cell] b10", flush=True)
man = {"created": time.strftime("%Y-%m-%d %H:%M"), "splits": {k: len(v["wav"]) for k, v in S.items()},
       "external": {k: [len(r.face), len(r.voice)] for k, r in EXT.items()}, "encoders": ENCODERS,
       "checks": CHECKS, "failed": FAILED, "audio": AUDIO_STATS, "broken": BROKEN, "resumed_arrays": len(SEEDED),
       "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "files": {f.name: list(np.load(f, mmap_mode="r").shape) for f in sorted(OUT.glob("*.npy"))}}
try:
    import speechbrain, insightface
    man["versions"].update(speechbrain=speechbrain.__version__, insightface=insightface.__version__)
except Exception:
    pass
man["reshape"] = RESHAPE                 # flattened wide arrays: row -> (L, D)
(OUT / "feats_manifest.json").write_text(json.dumps(man, indent=1))
shutil.rmtree(WORKDIR / "raw", ignore_errors=True)
tot = sum(p.stat().st_size for p in OUT.glob("*")) / 1e9
print(f"{len(man['files'])} arrays, {tot:.2f} GB in {OUT} | failed: {FAILED or 'none'}")
print("NEXT: Save Version (Save & Run All) -> Output -> New Dataset -> name it flag2027-feats-rfs-b-p (re-run with this output attached until every encoder is done)")

## RFS-01: aux rows (audio cache + frames)

In [ ]:
print("[cell] rfs aux", flush=True)
# VoxCeleb2 (vox = FLAG_37 rows, vox2b = FLAG_53 rows) and VoxBlink (vb = FLAG_46): voices from the audio cache, faces from the cached frames,
# each source in its own voice_meta.csv / face_meta.csv row order; rows of the other lane (and rows without audio / frame) stay NaN.
import threading
import concurrent.futures as cf
SMOKE = os.environ.get("FLAG_SMOKE") == "1"
LANE = "auxb"
PAR = 1 if LANE == "auxc" else 0
BUDGET = float(os.environ.get("RFS_BUDGET_H", "8.0")) * 3600
LIM = int(os.environ.get("RFS_AUX_MAX", "0")) or (24 if SMOKE else 0)
T_START = time.time()


def _inroots():
    base = Path("/kaggle/input")
    out = []
    for p in (base.iterdir() if base.exists() else []):
        if p.name in ("datasets", "notebooks", "kernels", "models"):
            out += [q for o in p.iterdir() if o.is_dir() for q in o.iterdir() if q.is_dir()]
        elif p.is_dir():
            out.append(p)
    return [p for p in out if not any(k in p.name for k in ("mavceleb", "raw-v4", "ext-meta"))]


_roots = _inroots()
_meta = {}
for r in _roots:
    for kind in ("voice", "face"):
        for p in r.rglob(f"{kind}_meta.csv"):
            for s, par in (("vox", "feats_vox2"), ("vox2b", "feats_vox2b"), ("vb", "feats_vb")):
                if p.parent.name == par:
                    _meta.setdefault((kind, s), p)
assert len(_meta) == 6, f"face / voice meta found for {sorted(_meta)}: attach lab-data / feats-vox2b / lab-vb"
_tar = sorted(q for r in _roots for q in r.rglob("tar0000.npz") if q.parent.name == "vox")
assert _tar, "attach duongphihoang/flag2027-audio-cache-p (vox/tar*.npz, vb/*.npz)"
ACROOT = _tar[0].parent.parent
CH = [(p, "vox") for p in sorted((ACROOT / "vox").glob("tar*.npz"))] + [(p, "vb") for p in sorted((ACROOT / "vb").glob("*.npz"))]
CH = CH[PAR::2]                                                 # b = even chunks, c = odd chunks (as FLAG_160)
dt = {"spk": str, "video": str, "utt": str}
VM = {s: pd.read_csv(_meta[("voice", s)], dtype=dt) for s in ("vox", "vox2b", "vb")}
FM = {s: pd.read_csv(_meta[("face", s)], dtype=dt) for s in ("vox", "vox2b", "vb")}
ROW = {s: {k: r for r, k in enumerate(zip(d.spk, d.video, d.utt))} for s, d in VM.items()}
log("audio cache", ACROOT, len(CH), "chunks of this lane | voice rows", {s: len(d) for s, d in VM.items()}, "| face rows", {s: len(d) for s, d in FM.items()})

# -- frames (FACELORA-03's layout): FLAG_37 rows in vox_frames_png.zip ("<row>.png"), FLAG_53 / VoxBlink in the FLAG_63 chunk zips ("spk/video/utt.png")
FILES = SCAN["files"]
_z37 = FILES.get("vox_frames_png.zip") or FILES.get("vox_frames_png.zipdata")
_zch = FILES.get("frames_manifest.json", [])
assert _z37 and _zch, "attach onnhqunhh/flag2027-lora-frames-p (vox_frames_png.zipdata + frames/)"
Z37, FR63 = sorted(_z37, key=lambda p: len(p.as_posix()))[0], _zch[0].parent
MEMBER = {}
with zipfile.ZipFile(Z37) as z:
    for n in z.namelist():
        MEMBER["v37/" + n] = Z37
for part in ("vox2b", "vb"):
    for zf in sorted([*(FR63 / part).glob("*.zip"), *(FR63 / part).glob("*.zipdata")]):
        with zipfile.ZipFile(zf) as z:
            for n in z.namelist():
                MEMBER[f"{part}/" + n] = zf
log(f"frames indexed: {sum(k.startswith('v37/') for k in MEMBER)} FLAG_37, {sum(k.startswith('vox2b/') for k in MEMBER)} FLAG_53, {sum(k.startswith('vb/') for k in MEMBER)} VoxBlink")
_ZT = threading.local()


def frame_bytes(member):
    zs = getattr(_ZT, "z", None)
    if zs is None:
        zs = _ZT.z = {}
    zp = MEMBER[member]
    if zp not in zs:
        zs[zp] = zipfile.ZipFile(zp)
    return zs[zp].read(member.split("/", 1)[1])


def member_of(s, i, r):
    return f"v37/{i}.png" if s == "vox" else f"{s}/{r.spk}/{r.video}/{r.utt}.png"


ND = torch.cuda.device_count()
D_F, D_V = ("cuda:0", "cuda:1" if ND > 1 else "cuda:0") if ND else ("cpu", "cpu")
CF, CV = load_core(D_F, "f"), load_core(D_V, "v")
FACE, VOICE = RFFace(CF, D_F), RFVoice(CV, D_V)
log("models ready: faces on", D_F, "| voices on", D_V)
A = {}                                                          # (kind, src, block) -> (N, d) float16
for s in ("vox", "vox2b", "vb"):
    for b, d in (("clip", 768), ("ibf", 1024)):
        A[("face", s, b)] = np.full((len(FM[s]), d), np.nan, np.float16)
    for b, d in (("sam", 256), ("r221", 256), ("xv", 192), ("ibv", 1024)):
        A[("voice", s, b)] = np.full((len(VM[s]), d), np.nan, np.float16)
DONE = {"face": 0, "voice": 0}
STOP = {"budget": False}


def face_job():
    log("[aux-face] start")
    n_all = sum(len(d) for d in FM.values()); t0 = time.time()
    with cf.ThreadPoolExecutor(8) as ex:
        for s in ("vox", "vox2b", "vb"):
            rows = [(i, r) for i, r in enumerate(FM[s].itertuples()) if i % 2 == PAR and member_of(s, i, r) in MEMBER]
            if LIM:
                rows = rows[:LIM]
            for b0 in range(0, len(rows), 64):
                if time.time() - T_START > BUDGET:
                    STOP["budget"] = True; return
                chunk = rows[b0:b0 + 64]
                xs = list(ex.map(lambda ir: FACE.prep(Image.open(io.BytesIO(frame_bytes(member_of(s, *ir)))).convert("RGB")), chunk))
                R = FACE.run(torch.stack(xs))
                idx = [i for i, _ in chunk]
                A[("face", s, "clip")][idx] = R["clip"]; A[("face", s, "ibf")][idx] = R[""]
                DONE["face"] += len(idx)
                if LIM or (b0 // 64) % 20 == 0:
                    log(f"[aux-face] {s} {b0 + len(idx)}/{len(rows)} (all {DONE['face']}) {DONE['face'] / max(time.time() - t0, 1):.1f} img/s")


def clips(Z, a, b):
    W = Z["wav"][a:b]
    ln = Z["n"][a:b].astype(int) if "n" in Z.files else [W.shape[1]] * len(W)
    return [W[i][:max(int(ln[i]), 16000 // 2)].astype(np.float32) / 32768.0 for i in range(len(W))]


def voice_job():
    log("[aux-voice] start")
    t0 = time.time()
    for ci, (p, kind) in enumerate(CH):
        Z = np.load(p)
        keys = [tuple(map(str, k)) for k in zip(Z["spk"], Z["video"], Z["utt"])]
        for c0 in range(0, len(keys), 64):
            if time.time() - T_START > BUDGET:
                STOP["budget"] = True; return
            for j, w in enumerate(clips(Z, c0, c0 + 64)):
                F_ = VOICE.feats(w)
                for s in (("vb",) if kind == "vb" else ("vox", "vox2b")):
                    r = ROW[s].get(keys[c0 + j])
                    if r is not None:
                        for b, k in (("ibv", ""), ("sam", "sam"), ("r221", "r221"), ("xv", "xv")):
                            A[("voice", s, b)][r] = F_[k]
                DONE["voice"] += 1
                if LIM:
                    log(f"[aux-voice] clip {DONE['voice']}")
                if LIM and DONE["voice"] >= LIM:
                    return
        log(f"[aux-voice] chunk {ci + 1}/{len(CH)}: {DONE['voice']} clips ({DONE['voice'] / max(time.time() - t0, 1):.1f} clips/s, {time.time() - t0:.0f}s)")


ERR = []
def guard(fn):
    def w():
        try:
            fn()
        except BaseException as e:
            import traceback; ERR.append(f"{fn.__name__}: {e!r}"); traceback.print_exc()
    return w


if D_F == D_V == "cpu":                                         # smoke on CPU: one job at a time (two jobs would fight for the same cores)
    guard(face_job)(); guard(voice_job)()
else:
    th = [threading.Thread(target=guard(face_job)), threading.Thread(target=guard(voice_job))]
    [t.start() for t in th]; [t.join() for t in th]
assert not ERR, ERR
for s in ("vox", "vox2b", "vb"):
    (OUT / s).mkdir(exist_ok=True)
    shutil.copy(_meta[("voice", s)], OUT / s / "voice_meta.csv"); shutil.copy(_meta[("face", s)], OUT / s / "face_meta.csv")
    for (kind, ss, b), X in A.items():
        if ss == s:
            np.save(OUT / s / f"{kind}_rf{b}.npy", X)
AUX_COVER = {f"{s}/{kind}_rf{b}": int(np.isfinite(X[:, 0].astype(np.float32)).sum()) for (kind, s, b), X in A.items()}
log(f"[aux] faces {DONE['face']}, voices {DONE['voice']} [{time.time() - T_START:.0f}s]; budget stop: {STOP['budget']}; rows per array {AUX_COVER}")
CHECKS["rfs01_aux"] = dict(cover=AUX_COVER, chunks=len(CH), lane=LANE, faces=DONE["face"], voices=DONE["voice"], budget_stop=STOP["budget"])

In [ ]:
print("[cell] upload", flush=True)
UPL = os.environ.get("RFS_UPLOAD", "")
(OUT / "rfs01_meta.json").write_text(json.dumps(dict(exp="RFS-01 step 1", lane="auxb", core_sha256="25abe24db6fbe863109dafd5f75f9075164dce75f77ce276695daf8687d338f7", dtype="float16 (aux, v1 / v2) / float32 (v4 train + dev lists)",
        blocks="face: rfclip 768, rfibf 1024, rfagef 10 (splits only); voice: rfsam 256, rfr221 256, rfxv 192, rfibv 1024, rfagev 4 (splits only)",
        rows="a: split / ext arrays in the FLAG_12 order (feats_manifest.json: splits, external); b, c: <vox|vox2b|vb>/{face,voice}_rf<block>.npy in that source's face_meta.csv / voice_meta.csv order (copied beside), NaN = not in this lane (faces: i % 2 == 0 in b, 1 in c; voices: even / odd chunks of the cache list)",
        voice_clip="aux: cached centre 3 s clip; MAV: FLAG_12 loader (whole file, v1 / v2 12 s centre crop)", faces="aux: cached 224 PNG frames",
        unreadable_file="a: framework stand-in (1 s silence / black image), listed in feats_manifest.json broken"), indent=1))
if UPL and os.environ.get("FLAG_SMOKE") != "1":
    try:
        import kagglehub
        kagglehub.dataset_upload(UPL, str(OUT), version_notes="RFS-01 step 1 lane auxb")
        print("uploaded ->", UPL, flush=True)
    except Exception as e:
        print(f"upload to {UPL} failed ({e!r}); the arrays are in the notebook output", flush=True)
print("ALL CELLS OK", flush=True)